#기본세팅

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.exists('/content/work/data/vessel_catalog.csv'):
    os.system('mkdir -p /content/work/data /content/work/model /content/work/output /content/raw /content/baseline')
    os.system('unzip -o -q /content/drive/MyDrive/vessel/open_data.zip -d /content/raw')
    os.system('unzip -o -q /content/raw/open.zip -d /content/work/data')
    os.system('unzip -o -q /content/raw/baseline_submit.zip -d /content/baseline')
print(os.listdir('/content/work/data'))

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

D = '/content/work/data'
train = pd.read_csv(f'{D}/train.csv')
cat   = pd.read_csv(f'{D}/vessel_catalog.csv')
sub   = pd.read_csv(f'{D}/sample_submission.csv')

print(train.shape, cat.shape, sub.shape)

In [ ]:
print('== train 컬럼')
print(train.dtypes)
print('\n== catalog 컬럼')
print(cat.dtypes)

display(train.head(10))
display(cat.head(10))

In [ ]:
m = train.merge(cat, left_on='target_vessel_id', right_on='vessel_id', suffixes=('_rec', '_cat'))

cols = ['vessel_name_rec', 'vessel_name_cat',
        'gross_tonnage_rec', 'gross_tonnage_cat',
        'flag_name_rec', 'flag_name_cat',
        'vessel_type_rec', 'vessel_type_cat']
print(m[cols].sample(20, random_state=0).to_string())

# 데이터 검증셋 분할(선박 단위,고정)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(gss.split(train, groups=train['target_vessel_id']))
tr, va = train.iloc[tr_idx], train.iloc[va_idx]

print('학습용:', len(tr), '/ 검증용:', len(va))
print('양쪽에 겹치는 선박 수:', len(set(tr['target_vessel_id']) & set(va['target_vessel_id'])))
print('검증용 중 호출부호로 찾을 수 있는 비율:', round(va['call_sign'].isin(set(tr['call_sign'])).mean(), 3))

is_kor = lambda s: s.astype(str).str.contains('[가-힣]')
print('한글 비율 - 학습용:', round(is_kor(tr['vessel_name']).mean(), 3),
      '/ 검증용:', round(is_kor(va['vessel_name']).mean(), 3))

# 여기 아래부턴 실험 진행 코드

In [ ]:
# 단어 사전 만드는 셀

from collections import Counter, defaultdict

id2name = dict(zip(cat['vessel_id'], cat['vessel_name']))

def ko_tokens(s):
    toks = str(s).split()
    # 마지막 단어 끝의 '호'(~號) 제거: '파이오니어호' -> '파이오니어'
    if toks and len(toks[-1]) > 1 and toks[-1].endswith('호'):
        toks[-1] = toks[-1][:-1]
    return toks

def en_tokens(s):
    return str(s).upper().split()

# 1) 학습용 한글 기록에서 단어 쌍 모으기
kor_tr = tr[is_kor(tr['vessel_name'])]
pair_count = Counter()
n_aligned = 0
for ko, vid in zip(kor_tr['vessel_name'], kor_tr['target_vessel_id']):
    k, e = ko_tokens(ko), en_tokens(id2name.get(vid, ''))
    if len(k) == len(e):
        n_aligned += 1
        pair_count.update(zip(k, e))
print(f'단어 수가 같아서 짝지을 수 있었던 기록: {n_aligned} / {len(kor_tr)}')

# 2) 한글 단어마다 가장 자주 짝지어진 영문 단어 하나를 고름
ko2en = defaultdict(Counter)
for (k, e), c in pair_count.items():
    ko2en[k][e] += c
word_dict = {k: cnt.most_common(1)[0][0] for k, cnt in ko2en.items()}
print('사전 단어 수:', len(word_dict))

print('\n자주 나온 쌍 상위 30개:')
for (k, e), c in pair_count.most_common(30):
    print(f'  {k} -> {e}  ({c}회)')

# 3) 검증용 한글 단어 중 사전에 있는 비율
kor_va = va[is_kor(va['vessel_name'])]
va_toks = [t for s in kor_va['vessel_name'] for t in ko_tokens(s)]
print('\n검증용 한글 단어 중 사전에 있는 비율:', round(sum(t in word_dict for t in va_toks) / len(va_toks), 3))

In [ ]:
!pip install -q rapidfuzz==3.14.3 Unidecode==1.4.0

import numpy as np
from rapidfuzz import process, fuzz
from unidecode import unidecode

def translate(name):
    out = []
    for t in ko_tokens(name):
        out.append(word_dict[t] if t in word_dict else unidecode(t).upper())
    return ' '.join(out)

cat_names = cat['vessel_name'].astype(str).str.upper().values
cat_gt    = cat['gross_tonnage'].values
cat_ids   = cat['vessel_id'].values
all_idx   = np.arange(len(cat))

def match(name, gt, tol=0.10):
    q = translate(name)
    idx = all_idx
    if pd.notna(gt):
        cand = np.where(np.abs(cat_gt - gt) <= gt * tol)[0]
        if len(cand) > 0:
            idx = cand
    best = process.extractOne(q, cat_names[idx], scorer=fuzz.ratio)
    return cat_ids[idx[best[2]]], q

kor_va = va[is_kor(va['vessel_name'])].copy()
gt_in = kor_va['gross_tonnage'].fillna(kor_va['international_gross_tonnage'])
res = [match(n, g) for n, g in zip(kor_va['vessel_name'], gt_in)]
kor_va['pred2'] = [r[0] for r in res]
kor_va['변환'] = [r[1] for r in res]

acc = (kor_va['pred2'] == kor_va['target_vessel_id']).mean()
print(f'한글 정확도: 베이스라인 0.374 → 사전 매칭 {acc:.3f}')

kor_va['정답_이름'] = kor_va['target_vessel_id'].map(id2name)
print(kor_va[['vessel_name', '변환', '정답_이름']].head(20).to_string())

In [ ]:
import re

VOW = set('AEIOUY')

# 한글 초성 19개, 종성 28개를 뼈대 자음으로
CHO  = ['K','K','N','T','T','R','M','P','P','S','S','','J','J','J','K','T','P','H']
JONG = ['','K','K','K','N','N','N','T','R','R','R','R','R','R','R','R',
        'M','P','P','T','T','N','T','T','K','T','P','']

def collapse(s):
    # 같은 글자가 연속되면 하나로: 'KSPRR' -> 'KSPR'
    return re.sub(r'(.)\1+', r'\1', s)

def ko_key(word):
    out = []
    for ch in str(word):
        if '가' <= ch <= '힣':
            code = ord(ch) - 0xAC00
            out.append(CHO[code // 588] + JONG[code % 28])
        elif ch.isdigit():
            out.append(ch)
    return collapse(''.join(out))

def en_key(word):
    s = re.sub(r'[^A-Z0-9]', '', str(word).upper())
    out, i, n = [], 0, len(s)
    while i < n:
        c, nx, two = s[i], s[i+1:i+2], s[i:i+2]
        if c.isdigit():                  out.append(c); i += 1
        elif two == 'PH':                out.append('P'); i += 2
        elif two in ('TH', 'SH'):        out.append('S'); i += 2
        elif two == 'CH':                out.append('J'); i += 2
        elif two == 'CK':                out.append('K'); i += 2
        elif two == 'NG' and s[i+2:i+3] not in VOW:
                                         out.append('N'); i += 2
        elif c in VOW or c == 'W':       i += 1
        elif c == 'H':                   out.append('H' if nx in VOW else ''); i += 1
        elif c == 'C':                   out.append('S' if nx and nx in 'EIY' else 'K'); i += 1
        elif c == 'G':                   out.append('J' if nx and nx in 'EI' else 'K'); i += 1
        elif c == 'X':                   out.append('KS'); i += 1
        elif c in 'QK':                  out.append('K'); i += 1
        elif c in 'BPVF':                out.append('P'); i += 1
        elif c in 'DT':                  out.append('T'); i += 1
        elif c in 'LR':                  out.append('R'); i += 1
        elif c in 'ZJ':                  out.append('J'); i += 1
        else:                            out.append(c); i += 1
    return collapse(''.join(out))

def query_key(name):
    # 사전에 있는 단어는 영문으로 바꾼 뒤 영문 규칙, 없는 단어는 한글 규칙
    parts = []
    for t in ko_tokens(name):
        parts.append(en_key(word_dict[t]) if t in word_dict else ko_key(t))
    return ''.join(parts)

for ko, en in [('네스토르', 'NESTOR'), ('헌터', 'HUNTER'), ('머큐리', 'MERCURY'),
               ('갤럭시', 'GALAXY'), ('익스플러어', 'EXPLORER'), ('제니스', 'JANICE')]:
    print(f'{ko:6s} {ko_key(ko):8s} | {en:9s} {en_key(en)}')

In [ ]:
import numpy as np
from rapidfuzz import process, fuzz
from unidecode import unidecode

def translate(name):
    return ' '.join(word_dict.get(t, unidecode(t).upper()) for t in ko_tokens(name))

cat_names = cat['vessel_name'].astype(str).str.upper().values
cat_keys  = np.array([en_key(n) for n in cat['vessel_name']])
cat_gt    = cat['gross_tonnage'].values
cat_ids   = cat['vessel_id'].values
all_idx   = np.arange(len(cat))

def predict_variant(name, gt, mode, tol=0.10):
    idx = all_idx
    if pd.notna(gt):
        cand = np.where(np.abs(cat_gt - gt) <= gt * tol)[0]
        if len(cand) > 0:
            idx = cand
    s_text = process.cdist([translate(name)], cat_names[idx], scorer=fuzz.ratio)[0]
    s_key  = process.cdist([query_key(name)], cat_keys[idx], scorer=fuzz.ratio)[0]
    score = {'text': s_text, 'key': s_key, 'sum': s_text + s_key}[mode]
    return cat_ids[idx[np.argmax(score)]]

kor_va = va[is_kor(va['vessel_name'])].copy()
gt_intl = kor_va['international_gross_tonnage'].fillna(kor_va['gross_tonnage'])

for mode in ['text', 'key', 'sum']:
    p = [predict_variant(n, g, mode) for n, g in zip(kor_va['vessel_name'], gt_intl)]
    kor_va[f'pred_{mode}'] = p
    acc = (np.array(p) == kor_va['target_vessel_id'].values).mean()
    print(f'{mode:5s}: {acc:.3f}')

#6.표기규칙+교차검증준비

In [ ]:
# ===== 표기 패턴 규칙 (제N, 알파벳 약어, 숫자 단어) =====
LETTERS = {'더블유':'W','에이치':'H','에이':'A','에프':'F','에스':'S','엑스':'X','제트':'Z',
           '제이':'J','케이':'K','브이':'V','와이':'Y','아이':'I','엘':'L','엠':'M','엔':'N',
           '앤':'N','알':'R','비':'B','씨':'C','시':'C','디':'D','이':'E','지':'G','오':'O',
           '피':'P','큐':'Q','티':'T','유':'U'}
L_KEYS = sorted(LETTERS, key=len, reverse=True)
NUM_WORD = {'투': 'II', '쓰리': 'III'}

def parse_letters(t):
    out, i = [], 0
    while i < len(t):
        for k in L_KEYS:
            if t.startswith(k, i):
                out.append(LETTERS[k]); i += len(k); break
        else:
            return None
    return ''.join(out) if len(out) >= 2 else None

def expand_tokens(name):
    out = []
    for t in ko_tokens(name):
        m = re.match(r'^제(\d+)(.*)$', t)
        if m:
            out.append('NO.' + m.group(1))
            if m.group(2):
                out.append(m.group(2))
        else:
            out.append(t)
    return out

def conv(t):
    if t in word_dict:             return word_dict[t], True
    if not re.search('[가-힣]', t):  return t.upper(), True
    if t in NUM_WORD:              return NUM_WORD[t], True
    L = parse_letters(t)
    if L:                          return L, True
    return unidecode(t).upper(), False

def translate2(name):
    return ' '.join(conv(t)[0] for t in expand_tokens(name))

def query_key2(name):
    parts = []
    for t in expand_tokens(name):
        text, is_en = conv(t)
        parts.append(en_key(text) if is_en else ko_key(t))
    return ''.join(parts)

# ===== 교차 검증 준비 =====
from sklearn.model_selection import GroupKFold

def build_dict(df):
    kor = df[is_kor(df['vessel_name'])]
    pc = Counter()
    for ko, vid in zip(kor['vessel_name'], kor['target_vessel_id']):
        k, e = ko_tokens(ko), en_tokens(id2name.get(vid, ''))
        if len(k) == len(e):
            pc.update(zip(k, e))
    k2e = defaultdict(Counter)
    for (k, e), c in pc.items():
        k2e[k][e] += c
    return {k: cnt.most_common(1)[0][0] for k, cnt in k2e.items()}

gkf = GroupKFold(n_splits=5)
print('준비 완료')

In [ ]:
cat_flag = cat['flag_name'].astype(str).values
cat_type = cat['vessel_type'].astype(str).values

def build_maps(df):
    m = df.merge(cat[['vessel_id', 'flag_name', 'vessel_type']],
                 left_on='target_vessel_id', right_on='vessel_id', suffixes=('', '_cat'))
    flag_map = {f: g.value_counts().idxmax() for f, g in m.groupby('flag_name')['flag_name_cat']}
    type_map = {t: g.value_counts(normalize=True).to_dict() for t, g in m.groupby('vessel_type')['vessel_type_cat']}
    return flag_map, type_map

def predict4(name, gt_intl, gt_dom, flag, vtype, w_flag, w_type):
    # 톤수 후보: submit_7 방식 (국제 ±10%, 없으면 국내 ±10%, 둘 다 없으면 전체)
    gt = gt_intl if pd.notna(gt_intl) else gt_dom
    idx = all_idx
    if pd.notna(gt):
        cand = np.where(np.abs(cat_gt - gt) <= gt * 0.10)[0]
        if len(cand) > 0:
            idx = cand
    s = process.cdist([translate2(name)], cat_names[idx], scorer=fuzz.ratio)[0] \
      + process.cdist([query_key2(name)], cat_keys[idx], scorer=fuzz.ratio)[0]
    # 국적 일치 가산점, 선종 확률 가산점
    target_flag = flag_map.get(flag)
    if target_flag is not None:
        s = s + w_flag * (cat_flag[idx] == target_flag)
    tp = type_map.get(vtype)
    if tp:
        s = s + w_type * pd.Series(cat_type[idx]).map(tp).fillna(0).values
    return cat_ids[idx[np.argmax(s)]]

def cv_korean4(no_gt, w_flag, w_type):
    global word_dict, flag_map, type_map
    hits = []
    for fold, (tri, vai) in enumerate(gkf.split(train, groups=train['target_vessel_id'])):
        trf = train.iloc[tri]
        word_dict = build_dict(trf)
        flag_map, type_map = build_maps(trf)
        kv = train.iloc[vai]
        kv = kv[is_kor(kv['vessel_name'])]
        gi = [np.nan] * len(kv) if no_gt else kv['international_gross_tonnage']
        gd = [np.nan] * len(kv) if no_gt else kv['gross_tonnage']
        pred = [predict4(n, a, b, f, t, w_flag, w_type)
                for n, a, b, f, t in zip(kv['vessel_name'], gi, gd, kv['flag_name'], kv['vessel_type'])]
        hits.extend(np.array(pred) == kv['target_vessel_id'].values)
    return np.mean(hits)

In [ ]:
from sklearn.model_selection import GroupKFold

# ----- 표기 규칙 -----
LETTERS = {'더블유':'W','에이치':'H','에이':'A','에프':'F','에스':'S','엑스':'X','제트':'Z',
           '제이':'J','케이':'K','브이':'V','와이':'Y','아이':'I','엘':'L','엠':'M','엔':'N',
           '앤':'N','알':'R','비':'B','씨':'C','시':'C','디':'D','이':'E','지':'G','오':'O',
           '피':'P','큐':'Q','티':'T','유':'U'}
L_KEYS = sorted(LETTERS, key=len, reverse=True)
NUM_WORD = {'투': 'II', '쓰리': 'III'}

def parse_letters(t):
    out, i = [], 0
    while i < len(t):
        for k in L_KEYS:
            if t.startswith(k, i):
                out.append(LETTERS[k]); i += len(k); break
        else:
            return None
    return ''.join(out) if len(out) >= 2 else None

def expand_tokens(name):
    out = []
    for t in ko_tokens(name):
        m = re.match(r'^제(\d+)(.*)$', t)
        if m:
            out.append('NO.' + m.group(1))
            if m.group(2):
                out.append(m.group(2))
        else:
            out.append(t)
    return out

def conv(t):
    if t in word_dict:             return word_dict[t], True
    if not re.search('[가-힣]', t):  return t.upper(), True
    if t in NUM_WORD:              return NUM_WORD[t], True
    L = parse_letters(t)
    if L:                          return L, True
    return unidecode(t).upper(), False

# ----- 번호 정리 + 쪼개기 -----
def norm_text(s):
    s = re.sub(r'(?<=[A-Z])(?=\d)|(?<=\d)(?=[A-Z])', ' ', str(s).upper())
    s = re.sub(r'\bNO\.?\s*(?=\d)', '', s)
    toks = s.split()
    return ' '.join([t for t in toks if not t.isdigit()] + [t for t in toks if t.isdigit()])

def num_last(k):
    return ''.join(c for c in k if not c.isdigit()) + ''.join(c for c in k if c.isdigit())

def segment(t, depth=0):
    if t in word_dict or depth > 3 or len(t) < 3:
        return [t]
    for i in range(len(t) - 1, 1, -1):
        head, tail = t[:i], t[i:]
        if head in word_dict or parse_letters(head):
            return [head] + segment(tail, depth + 1)
    return [t]

def tokens3(name):
    out = []
    for t in expand_tokens(name):
        if re.search('[가-힣]', t) and t not in word_dict:
            out.extend(segment(t))
        else:
            out.append(t)
    return out

def translate3(name):
    return norm_text(' '.join(conv(t)[0] for t in tokens3(name)))

def query_key3(name):
    parts = []
    for t in tokens3(name):
        text, is_en = conv(t)
        parts.append(en_key(norm_text(text)) if is_en else ko_key(t))
    return num_last(''.join(parts))

cat_names_n = np.array([norm_text(n) for n in cat['vessel_name']])
cat_keys_n  = np.array([num_last(en_key(n)) for n in cat_names_n])
cat_flag = cat['flag_name'].astype(str).values
cat_type = cat['vessel_type'].astype(str).values

# ----- 사전, 국적·선종 대응표 -----
def build_dict(df):
    kor = df[is_kor(df['vessel_name'])]
    pc = Counter()
    for ko, vid in zip(kor['vessel_name'], kor['target_vessel_id']):
        k, e = ko_tokens(ko), en_tokens(id2name.get(vid, ''))
        if len(k) == len(e):
            pc.update(zip(k, e))
    k2e = defaultdict(Counter)
    for (k, e), c in pc.items():
        k2e[k][e] += c
    return {k: cnt.most_common(1)[0][0] for k, cnt in k2e.items()}

def build_maps(df):
    m = df.merge(cat[['vessel_id', 'flag_name', 'vessel_type']],
                 left_on='target_vessel_id', right_on='vessel_id', suffixes=('', '_cat'))
    flag_map = {f: g.value_counts().idxmax() for f, g in m.groupby('flag_name')['flag_name_cat']}
    type_map = {t: g.value_counts(normalize=True).to_dict() for t, g in m.groupby('vessel_type')['vessel_type_cat']}
    return flag_map, type_map

# ----- 현재 제출 버전(submit_10)과 같은 한글 예측 -----
def predict5(name, gt_intl, gt_dom, flag, vtype, w_flag=10, w_type=20):
    gt = gt_intl if pd.notna(gt_intl) else gt_dom
    idx = all_idx
    if pd.notna(gt):
        cand = np.where(np.abs(cat_gt - gt) <= gt * 0.10)[0]
        if len(cand) > 0:
            idx = cand
    s = process.cdist([translate3(name)], cat_names_n[idx], scorer=fuzz.ratio)[0] \
      + process.cdist([query_key3(name)], cat_keys_n[idx], scorer=fuzz.ratio)[0]
    target_flag = flag_map.get(flag)
    if target_flag is not None:
        s = s + w_flag * (cat_flag[idx] == target_flag)
    tp = type_map.get(vtype)
    if tp:
        s = s + w_type * pd.Series(cat_type[idx]).map(tp).fillna(0).values
    return cat_ids[idx[np.argmax(s)]]

gkf = GroupKFold(n_splits=5)
print('실험용 함수 준비 완료')

In [ ]:
import random

# 한글 모음·자음 번호 (유니코드 순서): 비슷한 것끼리 묶음
VSIM = {1:[5], 5:[1], 0:[4], 4:[8, 0], 8:[4, 13], 13:[8, 18], 18:[13], 6:[12], 12:[6]}   # ㅐㅔ, ㅏㅓ, ㅓㅗ, ㅗㅜ, ㅜㅡ, ㅕㅛ
CSIM = {0:[15], 15:[0], 3:[16], 16:[3], 7:[17], 17:[7], 9:[10], 10:[9], 12:[14], 14:[12]}  # ㄱㅋ, ㄷㅌ, ㅂㅍ, ㅅㅆ, ㅈㅊ

def perturb(name, kind, rng):
    s = list(str(name))
    pos = [i for i, ch in enumerate(s) if '가' <= ch <= '힣']
    if kind == '띄어쓰기 제거':
        return str(name).replace(' ', '')
    if kind == '글자 빠짐':
        if len(pos) >= 4:
            del s[rng.choice(pos)]
        return ''.join(s)
    rng.shuffle(pos)
    for i in pos:
        code = ord(s[i]) - 0xAC00
        c, v, j = code // 588, (code % 588) // 28, code % 28
        if kind == '모음 변형' and v in VSIM:
            v = rng.choice(VSIM[v])
        elif kind == '자음 변형' and c in CSIM:
            c = rng.choice(CSIM[c])
        else:
            continue
        s[i] = chr(0xAC00 + c * 588 + v * 28 + j)
        break
    return ''.join(s)

def cv_noisy(kind):
    global word_dict, flag_map, type_map
    rng = random.Random(42)
    hits = []
    for fold, (tri, vai) in enumerate(gkf.split(train, groups=train['target_vessel_id'])):
        trf = train.iloc[tri]
        word_dict = build_dict(trf)
        flag_map, type_map = build_maps(trf)
        kv = train.iloc[vai]
        kv = kv[is_kor(kv['vessel_name'])]
        names = [n if kind == '없음' else perturb(n, kind, rng) for n in kv['vessel_name']]
        pred = [predict5(n, a, b, f, t) for n, a, b, f, t in
                zip(names, kv['international_gross_tonnage'], kv['gross_tonnage'], kv['flag_name'], kv['vessel_type'])]
        hits.extend(np.array(pred) == kv['target_vessel_id'].values)
    return np.mean(hits)

for kind in ['없음', '모음 변형', '자음 변형', '글자 빠짐', '띄어쓰기 제거']:
    print(f'{kind:8s}: 한글 정확도 {cv_noisy(kind):.3f}')

In [ ]:
def to_jamo(s):
    # '파이' -> 'ㅍㅏㅇㅣ' (자모 단위 문자열)
    out = []
    for ch in str(s):
        if '가' <= ch <= '힣':
            code = ord(ch) - 0xAC00
            c, v, j = code // 588, (code % 588) // 28, code % 28
            out.append(chr(0x1100 + c) + chr(0x1161 + v) + (chr(0x11A7 + j) if j else ''))
        else:
            out.append(ch)
    return ''.join(out)

FUZZY_CUT = None   # None이면 기존과 동일 (정확히 같아야 사전에서 찾음)

def refresh_dict_index():
    global dict_keys, dict_jamo
    dict_keys = [k for k in word_dict if len(k) >= 2]
    dict_jamo = [to_jamo(k) for k in dict_keys]

def conv(t):
    if t in word_dict:             return word_dict[t], True
    if not re.search('[가-힣]', t):  return t.upper(), True
    if t in NUM_WORD:              return NUM_WORD[t], True
    L = parse_letters(t)
    if L:                          return L, True
    # 새로 추가: 사전 단어 중 자모 기준으로 충분히 비슷한 게 있으면 그 단어로 간주
    if FUZZY_CUT and len(t) >= 2:
        hit = process.extractOne(to_jamo(t), dict_jamo, scorer=fuzz.ratio, score_cutoff=FUZZY_CUT)
        if hit:
            return word_dict[dict_keys[hit[2]]], True
    return unidecode(t).upper(), False

def cv_noisy2(kind):
    global word_dict, flag_map, type_map
    rng = random.Random(42)
    hits = []
    for fold, (tri, vai) in enumerate(gkf.split(train, groups=train['target_vessel_id'])):
        trf = train.iloc[tri]
        word_dict = build_dict(trf)
        refresh_dict_index()
        flag_map, type_map = build_maps(trf)
        kv = train.iloc[vai]
        kv = kv[is_kor(kv['vessel_name'])]
        names = [n if kind == '없음' else perturb(n, kind, rng) for n in kv['vessel_name']]
        pred = [predict5(n, a, b, f, t) for n, a, b, f, t in
                zip(names, kv['international_gross_tonnage'], kv['gross_tonnage'], kv['flag_name'], kv['vessel_type'])]
        hits.extend(np.array(pred) == kv['target_vessel_id'].values)
    return np.mean(hits)

In [ ]:
FUZZY_CUT = 85    # submit_12 설정
KEY_L = 6         # submit_12 설정
W_TOK = 0         # 단어 단위 비교 점수의 비중 (0이면 submit_12와 동일)

def perturb2(name, kind, rng):
    if kind == '단어 빠짐':
        toks = str(name).split()
        if len(toks) >= 2:
            del toks[rng.randrange(len(toks))]
        return ' '.join(toks)
    return perturb(name, kind, rng)

def predict7(name, gt_intl, gt_dom, flag, vtype, w_flag=10, w_type=20):
    gt = gt_intl if pd.notna(gt_intl) else gt_dom
    idx = all_idx
    if pd.notna(gt):
        cand = np.where(np.abs(cat_gt - gt) <= gt * 0.10)[0]
        if len(cand) > 0:
            idx = cand
    q, qk = translate3(name), query_key3(name)
    w_key = 1.0 if KEY_L is None else min(1.0, len(qk) / KEY_L)
    s = process.cdist([q], cat_names_n[idx], scorer=fuzz.ratio)[0] \
      + w_key * process.cdist([qk], cat_keys_n[idx], scorer=fuzz.ratio)[0]
    if W_TOK:
        s = s + W_TOK * process.cdist([q], cat_names_n[idx], scorer=fuzz.token_set_ratio)[0]
    target_flag = flag_map.get(flag)
    if target_flag is not None:
        s = s + w_flag * (cat_flag[idx] == target_flag)
    tp = type_map.get(vtype)
    if tp:
        s = s + w_type * pd.Series(cat_type[idx]).map(tp).fillna(0).values
    return cat_ids[idx[np.argmax(s)]]

def cv_noisy4(kind):
    global word_dict, flag_map, type_map
    rng = random.Random(42)
    hits = []
    for fold, (tri, vai) in enumerate(gkf.split(train, groups=train['target_vessel_id'])):
        trf = train.iloc[tri]
        word_dict = build_dict(trf)
        refresh_dict_index()
        flag_map, type_map = build_maps(trf)
        kv = train.iloc[vai]
        kv = kv[is_kor(kv['vessel_name'])]
        names = [n if kind == '없음' else perturb2(n, kind, rng) for n in kv['vessel_name']]
        pred = [predict7(n, a, b, f, t) for n, a, b, f, t in
                zip(names, kv['international_gross_tonnage'], kv['gross_tonnage'], kv['flag_name'], kv['vessel_type'])]
        hits.extend(np.array(pred) == kv['target_vessel_id'].values)
    return np.mean(hits)

for w in [0, 0.5, 1.0]:
    W_TOK = w
    res = [f'{kind} {cv_noisy4(kind):.3f}' for kind in ['없음', '글자 빠짐', '단어 빠짐']]
    print(f'단어 비교 비중 {w}: ' + ' / '.join(res))

In [ ]:
W_TOK = 0
for wf, wt in [(10, 20), (15, 30), (20, 40)]:
    def predict7_w(name, a, b, f, t, _wf=wf, _wt=wt):
        return predict7(name, a, b, f, t, w_flag=_wf, w_type=_wt)
    _orig = predict7
    globals()['predict7'] = lambda n, a, b, f, t, w_flag=wf, w_type=wt: _orig(n, a, b, f, t, w_flag, w_type)
    res = [f'{kind} {cv_noisy4(kind):.3f}' for kind in ['없음', '글자 빠짐']]
    globals()['predict7'] = _orig
    print(f'국적 {wf} / 선종 {wt}: ' + ' / '.join(res))

#submit 파일 생성


In [ ]:
import json

!rm -rf /content/submit_v13 && mkdir -p /content/submit_v13/model

word_dict_full = build_dict(train)
flag_map_full, type_map_full = build_maps(train)

with open('/content/submit_v13/model/word_dict.json', 'w', encoding='utf-8') as f:
    json.dump(word_dict_full, f, ensure_ascii=False)
with open('/content/submit_v13/model/maps.json', 'w', encoding='utf-8') as f:
    json.dump({'flag': flag_map_full, 'type': type_map_full}, f, ensure_ascii=False)
with open('/content/submit_v13/requirements.txt', 'w') as f:
    f.write('# Uses the evaluation image\'s preinstalled numpy, pandas, rapidfuzz and Unidecode.\n')

print('사전 단어 수:', len(word_dict_full), '/ 국적 대응:', len(flag_map_full), '/ 선종 대응:', len(type_map_full))

In [ ]:
%%writefile /content/submit_v13/model/matcher.py
"""Vessel matcher.

Korean names: token rules + compound splitting + dictionary (exact, then fuzzy by jamo)
              -> text + consonant-skeleton similarity (numbers normalized, moved to end)
              + flag match bonus + vessel-type probability bonus
Non-Korean names: text similarity
Tonnage filter: international GT +-10%, else domestic GT +-10%, else whole catalog.
"""

import json
import re

import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from unidecode import unidecode

# ===== 설정 =====
W_FLAG, W_TYPE = 15, 30
FUZZY_CUT = 85      # 사전 유사 검색 기준 (자모 기준 유사도)
KEY_L = 6           # 짧은 뼈대 비중 축소 기준 길이 (None이면 사용 안 함)

HANGUL = re.compile("[가-힣]")
VOW = set("AEIOUY")
CHO = ["K","K","N","T","T","R","M","P","P","S","S","","J","J","J","K","T","P","H"]
JONG = ["","K","K","K","N","N","N","T","R","R","R","R","R","R","R","R",
        "M","P","P","T","T","N","T","T","K","T","P",""]
LETTERS = {"더블유":"W","에이치":"H","에이":"A","에프":"F","에스":"S","엑스":"X","제트":"Z",
           "제이":"J","케이":"K","브이":"V","와이":"Y","아이":"I","엘":"L","엠":"M","엔":"N",
           "앤":"N","알":"R","비":"B","씨":"C","시":"C","디":"D","이":"E","지":"G","오":"O",
           "피":"P","큐":"Q","티":"T","유":"U"}
L_KEYS = sorted(LETTERS, key=len, reverse=True)
NUM_WORD = {"투": "II", "쓰리": "III"}


def is_korean(name):
    return bool(HANGUL.search(str(name)))


def ko_tokens(s):
    toks = str(s).split()
    if toks and len(toks[-1]) > 1 and toks[-1].endswith("호"):
        toks[-1] = toks[-1][:-1]
    return toks


def expand_tokens(name):
    out = []
    for t in ko_tokens(name):
        m = re.match(r"^제(\d+)(.*)$", t)
        if m:
            out.append("NO." + m.group(1))
            if m.group(2):
                out.append(m.group(2))
        else:
            out.append(t)
    return out


def parse_letters(t):
    out, i = [], 0
    while i < len(t):
        for k in L_KEYS:
            if t.startswith(k, i):
                out.append(LETTERS[k]); i += len(k); break
        else:
            return None
    return "".join(out) if len(out) >= 2 else None


def to_jamo(s):
    out = []
    for ch in str(s):
        if "가" <= ch <= "힣":
            code = ord(ch) - 0xAC00
            c, v, j = code // 588, (code % 588) // 28, code % 28
            out.append(chr(0x1100 + c) + chr(0x1161 + v) + (chr(0x11A7 + j) if j else ""))
        else:
            out.append(ch)
    return "".join(out)


def norm_text(s):
    s = re.sub(r"(?<=[A-Z])(?=\d)|(?<=\d)(?=[A-Z])", " ", str(s).upper())
    s = re.sub(r"\bNO\.?\s*(?=\d)", "", s)
    toks = s.split()
    return " ".join([t for t in toks if not t.isdigit()] + [t for t in toks if t.isdigit()])


def num_last(k):
    return "".join(c for c in k if not c.isdigit()) + "".join(c for c in k if c.isdigit())


def collapse(s):
    return re.sub(r"(.)\1+", r"\1", s)


def ko_key(word):
    out = []
    for ch in str(word):
        if "가" <= ch <= "힣":
            code = ord(ch) - 0xAC00
            out.append(CHO[code // 588] + JONG[code % 28])
        elif ch.isdigit():
            out.append(ch)
    return collapse("".join(out))


def en_key(word):
    s = re.sub(r"[^A-Z0-9]", "", str(word).upper())
    out, i, n = [], 0, len(s)
    while i < n:
        c, nx, two = s[i], s[i+1:i+2], s[i:i+2]
        if c.isdigit():                  out.append(c); i += 1
        elif two == "PH":                out.append("P"); i += 2
        elif two in ("TH", "SH"):        out.append("S"); i += 2
        elif two == "CH":                out.append("J"); i += 2
        elif two == "CK":                out.append("K"); i += 2
        elif two == "NG" and s[i+2:i+3] not in VOW:
                                         out.append("N"); i += 2
        elif c in VOW or c == "W":       i += 1
        elif c == "H":                   out.append("H" if nx in VOW else ""); i += 1
        elif c == "C":                   out.append("S" if nx and nx in "EIY" else "K"); i += 1
        elif c == "G":                   out.append("J" if nx and nx in "EI" else "K"); i += 1
        elif c == "X":                   out.append("KS"); i += 1
        elif c in "QK":                  out.append("K"); i += 1
        elif c in "BPVF":                out.append("P"); i += 1
        elif c in "DT":                  out.append("T"); i += 1
        elif c in "LR":                  out.append("R"); i += 1
        elif c in "ZJ":                  out.append("J"); i += 1
        else:                            out.append(c); i += 1
    return collapse("".join(out))


class VesselMatcher:
    def __init__(self, catalog, dict_path, maps_path, tol=0.10):
        with open(dict_path, encoding="utf-8") as f:
            self.word_dict = json.load(f)
        with open(maps_path, encoding="utf-8") as f:
            maps = json.load(f)
        self.flag_map, self.type_map = maps["flag"], maps["type"]
        self.dict_keys = [k for k in self.word_dict if len(k) >= 2]
        self.dict_jamo = [to_jamo(k) for k in self.dict_keys]
        self.names = catalog["vessel_name"].astype(str).str.upper().values
        self.names_n = np.array([norm_text(n) for n in catalog["vessel_name"]])
        self.keys_n = np.array([num_last(en_key(n)) for n in self.names_n])
        self.gt = catalog["gross_tonnage"].values
        self.flags = catalog["flag_name"].astype(str).values
        self.types = catalog["vessel_type"].astype(str).values
        self.ids = catalog["vessel_id"].values
        self.all_idx = np.arange(len(catalog))
        self.tol = tol

    def conv(self, t):
        if t in self.word_dict:
            return self.word_dict[t], True
        if not HANGUL.search(t):
            return t.upper(), True
        if t in NUM_WORD:
            return NUM_WORD[t], True
        letters = parse_letters(t)
        if letters:
            return letters, True
        if FUZZY_CUT and len(t) >= 2:
            hit = process.extractOne(to_jamo(t), self.dict_jamo, scorer=fuzz.ratio, score_cutoff=FUZZY_CUT)
            if hit:
                return self.word_dict[self.dict_keys[hit[2]]], True
        return unidecode(t).upper(), False

    def segment(self, t, depth=0):
        if t in self.word_dict or depth > 3 or len(t) < 3:
            return [t]
        for i in range(len(t) - 1, 1, -1):
            head, tail = t[:i], t[i:]
            if head in self.word_dict or parse_letters(head):
                return [head] + self.segment(tail, depth + 1)
        return [t]

    def tokens(self, name):
        out = []
        for t in expand_tokens(name):
            if HANGUL.search(t) and t not in self.word_dict:
                out.extend(self.segment(t))
            else:
                out.append(t)
        return out

    def translate(self, name):
        return norm_text(" ".join(self.conv(t)[0] for t in self.tokens(name)))

    def query_key(self, name):
        parts = []
        for t in self.tokens(name):
            text, is_en = self.conv(t)
            parts.append(en_key(norm_text(text)) if is_en else ko_key(t))
        return num_last("".join(parts))

    def predict_one(self, name, gt_intl, gt_dom, flag, vtype):
        gt = gt_intl if pd.notna(gt_intl) else gt_dom
        idx = self.all_idx
        if pd.notna(gt):
            cand = np.where(np.abs(self.gt - gt) <= gt * self.tol)[0]
            if len(cand) > 0:
                idx = cand

        if not is_korean(name):
            best = process.extractOne(str(name).upper(), self.names[idx], scorer=fuzz.ratio)
            return self.ids[idx[best[2]]]

        qk = self.query_key(name)
        w_key = 1.0 if KEY_L is None else min(1.0, len(qk) / KEY_L)
        s = process.cdist([self.translate(name)], self.names_n[idx], scorer=fuzz.ratio)[0] \
          + w_key * process.cdist([qk], self.keys_n[idx], scorer=fuzz.ratio)[0]
        target_flag = self.flag_map.get(flag) if pd.notna(flag) else None
        if target_flag is not None:
            s = s + W_FLAG * (self.flags[idx] == target_flag)
        tp = self.type_map.get(vtype) if pd.notna(vtype) else None
        if tp:
            s = s + W_TYPE * pd.Series(self.types[idx]).map(tp).fillna(0).values
        return self.ids[idx[np.argmax(s)]]

In [ ]:
%%writefile /content/submit_v13/script.py
"""Vessel matching inference: name similarity + tonnage filter + flag/type bonus (Korean)."""

from pathlib import Path
import sys

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent / "model"))
from matcher import VesselMatcher


def run(
    test_path="data/test.csv",
    catalog_path="data/vessel_catalog.csv",
    dict_path="model/word_dict.json",
    maps_path="model/maps.json",
    output_path="output/submission.csv",
):
    test = pd.read_csv(test_path, encoding="utf-8-sig")
    catalog = pd.read_csv(catalog_path, encoding="utf-8-sig")
    if test["sample_id"].isna().any() or not test["sample_id"].is_unique:
        raise ValueError("test.csv sample_id must be non-null and unique")

    matcher = VesselMatcher(catalog, dict_path, maps_path)
    predictions = [
        matcher.predict_one(n, gi, gd, fl, vt)
        for n, gi, gd, fl, vt in zip(
            test["vessel_name"], test["international_gross_tonnage"], test["gross_tonnage"],
            test["flag_name"], test["vessel_type"],
        )
    ]

    if len(predictions) != len(test) or not set(predictions).issubset(set(catalog["vessel_id"])):
        raise ValueError("Invalid catalog predictions")
    result = pd.DataFrame({"sample_id": test["sample_id"], "target_vessel_id": predictions})
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Saved {len(result)} predictions to {output_path}")
    return output_path


if __name__ == "__main__":
    run()

In [ ]:
!rm -rf /content/test_run && cp -r /content/submit_v13 /content/test_run
!mkdir -p /content/test_run/data
!cp /content/work/data/vessel_catalog.csv /content/test_run/data/
va.drop(columns=['target_vessel_id']).to_csv('/content/test_run/data/test.csv', index=False)

!cd /content/test_run && time python script.py
!head -5 /content/test_run/output/submission.csv

In [ ]:
!cd /content/submit_v13 && rm -f /content/submit_13.zip && zip -r /content/submit_13.zip script.py requirements.txt model
!unzip -l /content/submit_13.zip

from google.colab import files
files.download('/content/submit_13.zip')